# Phase A — pilot diagnostics (no training) · Fokker–Planck consistency
Base model: the class-conditional MONAI DDPM of **U24_synthesis** (Linardos, Thakur, Gastounioti, Makris & Bakas,
*Evaluating the utility of AI-generated mammography images in breast cancer analysis*, Scientific Reports 2026), retrained
by us on CBIS-DDSM in Experiment 2. Nothing is trained here: every test uses the Experiment-2 checkpoint
`base2_w_ep040.pt` (100 % data, epoch 40) or the real images.

| test | what it decides | device |
|---|---|---|
| **A1** leakage / duplicates (HARD GATE) | patient or image overlap between train and test → FAIL | CPU |
| **A2** conditional-mean ("Jensen") test | is the old physics loss on x̂0 measuring blur? → sets **t\*** for the NLF arm | GPU 1 |
| **A3** noise vs texture + lesion size | is the "noise curve" noise or tissue texture at 256 px? | CPU |
| **A4** Fokker–Planck residual estimator | is the FP residual measurable, and is fp16 finite-difference precise enough? | GPU 0 |
| **A5** compute benchmark | cost of the FP term → number of seeds (≤2× → 5, ≤3× → 4, >3× → reconsider) | GPU 0 (after A4) |

Every rule is written in the script header **before** the run. The report only applies those rules; the LOCK is written
afterwards with the RA.

**How to run**
1. Settings → Accelerator **GPU T4 ×2** · Internet **ON**.
2. Add Input → *CBIS-DDSM Breast Cancer Image Dataset*.
3. Add Input → *Your Work* → **notebook64a7f79592** (the Experiment 2 notebook — it holds `base2_w_ep040.pt` and `base2_last.pt`).
4. Optional: Add Input → *Your Work* → **notebook00f3867662** (Experiment 3). Then A2 is also run on a 25 %-data model (+ ~1 h).
5. **Save Version → Save & Run All (Commit)**. Nothing to edit.

**Time:** probably 2–4 h (+ ~1 h if the 25 % model is attached). This is an estimate from Experiment 2's measured training speed
(1.67 it/s), not a measurement of these scripts; the logs print the real time of every step. Kaggle's 12 h limit is not at risk.

**Send back:** `phaseA/phaseA_report.txt` from the Output tab (or the whole `phaseA_results.zip`). If a cell fails, also send its log
from `phaseA/logs/`.

In [ ]:
# ===================== CELL 1 — CONFIG (nothing needs to be changed) =====================
RUN = {"A1": True, "A2": True, "A3": True, "A4": True, "A5": True}
RUN_A2_25 = True          # A2 also on a 25 %-data model - only if the Experiment 3 or 4 output is attached (+ ~1 h)
import os, time
if not os.path.exists("/tmp/pg_session_start"):
    open("/tmp/pg_session_start", "w").write(str(time.time()))
def used_h():
    return (time.time() - float(open("/tmp/pg_session_start").read())) / 3600
print("Phase A | tests:", [k for k, v in RUN.items() if v], "| A2 on 25 % model:", RUN_A2_25)

In [ ]:
# ===================== CELL 2 — setup (needs Internet ON) =====================
import os, re, sys, subprocess
os.chdir("/kaggle/working")
if not os.path.isdir("/kaggle/working/U24_synthesis/.git"):
    subprocess.run("git clone -q https://github.com/Its-Me-Sk/U24_synthesis.git", shell=True, check=True)
if not os.path.exists("/tmp/pip_done"):
    # Kaggle runs Python 3.13: install the repo requirements once, never force-reinstall numpy / pandas
    subprocess.run("cd /kaggle/working/U24_synthesis && grep -v xformers requirements.txt > /tmp/req.txt && "
                   "pip install -q -r /tmp/req.txt", shell=True, check=True)
    subprocess.run("pip install -q --no-deps torch-fidelity", shell=True, check=True)
    open("/tmp/pip_done", "w").write("ok")
probe = ("import sys, numpy, pandas, scipy, torch, torchmetrics, monai, generative, PIL; "
         "print('python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__, "
         "'| torch', torch.__version__, '| torchmetrics', torchmetrics.__version__, '| monai', monai.__version__)")
env_chk = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
if env_chk.returncode != 0:
    print(env_chk.stderr[-2500:])
    raise SystemExit("!! environment check failed (see the error above) -> send Claude this block")
print("environment OK:", env_chk.stdout.strip())
p = "/kaggle/working/U24_synthesis/lightning_synthesis/model_architectures/monai_ddpm.py"
s = open(p).read()
s2 = re.sub(r"use_flash_attention\s*=\s*True", "use_flash_attention=False", s)   # xformers is not installed
if s2 != s:
    open(p, "w").write(s2)
print("flash attention ->", "patched off" if s2 != s else "already off")
m = "/kaggle/working/U24_synthesis/models.py"          # oracle loader (used later in A6): hard-coded config path
s = open(m).read()
s2 = re.sub(r"open\(\s*['\"]/home/[^'\"]*config\.yaml['\"]", "open('/kaggle/working/U24_synthesis/config.yaml'", s)
if s2 != s:
    open(m, "w").write(s2)
print("Cell 2 done. Kernel restart NOT needed (all GPU work runs in subprocesses).")

In [ ]:
# ===================== CELL 3 — code + data cache + input checks + job runner =====================
import os, sys, glob, json, subprocess, threading, time
from pathlib import Path
WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
os.chdir(WD)
Path("phaseA_common.py").write_text(r'''# phaseA_common.py - shared helpers for the Phase A diagnostics (no training happens in Phase A).
# Model, schedule, data domain and conditioning are taken from the Experiment-2 code (train_arm2.py / eval_arm.py):
#   * model: model_architectures.MonaiDDPM().unet (MONAI-Generative DiffusionModelUNet, flash attention off)
#   * schedule: MonaiDDPM().scheduler.alphas_cumprod (linear beta 1e-4..0.02, T=1000)
#   * images: per-image min-max to [-1,1]; physics statistic on (x+1)/2
#   * conditioning: context = label as float (B,1,1); -1 = null label (classifier-free guidance)
import os, sys, glob, json, math, time
import numpy as np
import torch

WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
if WD not in sys.path:
    sys.path.insert(0, WD)
OUT = "/kaggle/working/phaseA"
os.makedirs(OUT, exist_ok=True)


def log(tag, *m):
    print(f"[{tag}]", *m, flush=True)


_INDEX = None


def find(name, roots=("/kaggle/input", "/kaggle/working")):
    """all files with this exact name in the attached inputs (one directory walk per process; image folders skipped)"""
    global _INDEX
    if _INDEX is None:
        _INDEX = {}
        for r in roots:
            for dp, dns, fns in os.walk(r):
                dns[:] = [d for d in dns if d not in ("jpeg", "U24_synthesis", ".git", "phaseA")]
                for fn in fns:
                    _INDEX.setdefault(fn, []).append(os.path.join(dp, fn))
    return sorted(set(_INDEX.get(name, [])))


def main_ckpt():
    """full-data Experiment-2 baseline, epoch 40 (the main Phase A checkpoint)"""
    h = find("base2_w_ep040.pt")
    if not h:
        raise SystemExit("base2_w_ep040.pt not found: Add Input -> Your Work -> the Experiment 2 notebook "
                         "(notebook64a7f79592), then run again")
    return h[0]


def optional_25_ckpt():
    """any 25 %-data baseline at epoch 45 from Experiment 3 or 4 (optional)"""
    for s in (42, 43, 44, 45, 46, 47, 48):
        h = find(f"base3_s{s}_w_ep045.pt")
        if h:
            return h[0]
    return None


def schedule(dev):
    from model_architectures import MonaiDDPM
    m = MonaiDDPM(lr=1e-5, T=1000)
    ac = m.scheduler.alphas_cumprod.detach().double().to(dev)
    unet = m.unet
    del m
    return unet, ac


def load_model(path, dev, dtype=torch.float32):
    """returns (unet in eval mode, alphas_cumprod float64, snapshot meta)"""
    snap = torch.load(path, map_location="cpu", weights_only=False)
    unet, ac = schedule(dev)
    sd = snap["unet"] if "unet" in snap else snap["models"]["net"]
    unet.load_state_dict({k: v.float() for k, v in sd.items()})
    unet = unet.to(dev).to(dtype).eval()
    for p in unet.parameters():
        p.requires_grad_(False)
    meta = {k: snap[k] for k in ("epoch", "step", "lam", "tau") if k in snap}
    meta["nlf"] = snap.get("nlf")
    meta["path"] = path
    del snap
    return unet, ac, meta


def load_cache(split):
    d = np.load(f"/tmp/{split}_cache.npz")
    return d["x"], d["y"]


def minmax_pm1(x):
    """x: torch (B,1,H,W) any range -> per-image min-max to [-1,1] (training domain)"""
    mn = x.amin(dim=(1, 2, 3), keepdim=True)
    mx = x.amax(dim=(1, 2, 3), keepdim=True)
    return 2 * (x - mn) / (mx - mn + 1e-8) - 1


def to_model_domain(x_u8, dev):
    return minmax_pm1(torch.from_numpy(x_u8).float().unsqueeze(1).to(dev))


def ctx_of(labels, dev, dtype=torch.float32):
    """labels: list/array of 0, 1 or -1 (null) -> context tensor (B,1,1)"""
    return torch.as_tensor(np.asarray(labels, dtype=np.float32), device=dev).view(-1, 1, 1).to(dtype)


def eps_pred(unet, x, t_int, ctx, amp=False, chunk=8):
    """epsilon prediction. amp=True: exactly as in training (fp32 weights + fp16 autocast); amp=False: the weights'
    own dtype. Large batches are split into chunks (the non-flash attention at 64x64 needs ~0.5 GB per image)."""
    if chunk and x.shape[0] > chunk:
        if ctx.shape[0] == 1:
            ctx = ctx.expand(x.shape[0], *ctx.shape[1:])
        return torch.cat([eps_pred(unet, x[i:i + chunk], t_int, ctx[i:i + chunk], amp, None)
                          for i in range(0, x.shape[0], chunk)])
    tb = torch.full((x.shape[0],), int(t_int), device=x.device, dtype=torch.long)
    if amp:
        with torch.autocast("cuda", dtype=torch.float16):
            out = unet(x, timesteps=tb, context=ctx)
        return out.float()
    return unet(x, timesteps=tb, context=ctx)


def save_json(obj, name):
    with open(os.path.join(OUT, name), "w") as fh:
        json.dump(obj, fh, indent=1, default=float)
''')
Path("pinn_physics.py").write_text(r'''# pinn_physics.py - physics-informed soft constraint (PINN-style residual) for mammogram DDPM.
# Physical model: signal-dependent noise-level function (NLF) of the X-ray film/digitiser chain
#   sigma^2(mu) = b + a*mu + c*mu^2
#   b: Gaussian read/digitiser noise, a*mu: Poisson (quantum) noise, c*mu^2: multiplicative film-grain/gain term
# The coefficients are CALIBRATED on real training images, then the residual between the NLF of a
# generated image and the physical NLF is added to the diffusion loss (lambda * residual).
import torch
import torch.nn.functional as F


class PhysicsNLF:
    def __init__(self, device, k=7, K=16, lo=0.08, hi=0.95, mask_lo=0.04):
        ax = torch.arange(7, device=device, dtype=torch.float32) - 3
        g = torch.exp(-ax ** 2 / 2.0)
        self.g = g / g.sum()
        self.k, self.K, self.lo, self.hi, self.mask_lo = k, K, lo, hi, mask_lo
        self.c = torch.linspace(lo, hi, K, device=device)
        self.bw = float(hi - lo) / (K - 1) * 0.6
        self.p = None                     # (b, a, c) of the calibrated NLF

    def blur(self, I):
        g = self.g
        I = F.conv2d(F.pad(I, (3, 3, 0, 0), mode="reflect"), g.view(1, 1, 1, 7))
        return F.conv2d(F.pad(I, (0, 0, 3, 3), mode="reflect"), g.view(1, 1, 7, 1))

    def stats(self, I):
        # I: (B,1,H,W) intensities in [0,1]. Returns per-image binned high-frequency variance vs local mean.
        h = I - self.blur(I)
        mu = F.avg_pool2d(I, self.k, 1, self.k // 2, count_include_pad=False).flatten(1)
        v = F.avg_pool2d(h * h, self.k, 1, self.k // 2, count_include_pad=False).flatten(1)
        W = torch.exp(-(mu.unsqueeze(-1) - self.c) ** 2 / (2 * self.bw ** 2))
        W = W * (mu > self.mask_lo).unsqueeze(-1).float()        # ignore black background (air)
        mass = W.sum(1)                                           # (B,K)
        vbin = torch.einsum("bpk,bp->bk", W, v) / (mass + 1e-6)   # (B,K)
        return vbin, mass

    def model_var(self):
        b, a, c = self.p
        return (b + a * self.c + c * self.c ** 2).clamp_min(1e-7)

    def residual(self, I):
        # per-image physics residual: mass-weighted squared log-ratio of measured vs physical NLF
        vbin, mass = self.stats(I)
        r = (torch.log(vbin + 1e-5) - torch.log(self.model_var() + 1e-5)) ** 2
        w = mass / (mass.sum(1, keepdim=True) + 1e-6)
        return (w * r).sum(1)

    @torch.no_grad()
    def fit(self, X_u8, bs=32):
        dev = self.c.device
        tv = torch.zeros(self.K, device=dev, dtype=torch.float64)
        tm = torch.zeros(self.K, device=dev, dtype=torch.float64)
        for i in range(0, X_u8.shape[0], bs):
            I = X_u8[i:i + bs].to(dev).float().div(255).unsqueeze(1)
            vb, m = self.stats(I)
            tv += (vb * m).sum(0).double()
            tm += m.sum(0).double()
        V = tv / tm.clamp_min(1e-6)
        wt = tm / tm.sum()
        c = self.c.double()

        def wls(cols):
            A = torch.stack(cols, 1)
            th = torch.linalg.solve(A.T @ (wt[:, None] * A), A.T @ (wt * V))
            pred = (A @ th).clamp_min(1e-9)
            ss_res = (wt * (V - pred) ** 2).sum()
            ss_tot = (wt * (V - (wt * V).sum()) ** 2).sum()
            lr = torch.log(V) - torch.log(pred)
            lres = (wt * lr ** 2).sum()
            ltot = (wt * (torch.log(V) - (wt * torch.log(V)).sum()) ** 2).sum()
            return th, float(1 - ss_res / ss_tot), float(1 - lres / ltot)

        th_lin, r2_lin, lr2_lin = wls([torch.ones_like(c), c])
        th_q, r2_q, lr2_q = wls([torch.ones_like(c), c, c ** 2])
        self.p = (float(th_q[0]), float(th_q[1]), float(th_q[2]))
        return {"model": "sigma^2 = b + a*mu + c*mu^2 (Poisson-Gaussian + multiplicative)",
                "b": self.p[0], "a": self.p[1], "c": self.p[2],
                "R2_var": r2_q, "R2_logvar": lr2_q,
                "linear_poisson_gaussian_only": {"b": float(th_lin[0]), "a": float(th_lin[1]),
                                                  "R2_var": r2_lin, "R2_logvar": lr2_lin},
                "bin_centers": [float(x) for x in c], "measured_var": [float(x) for x in V],
                "bin_weight": [float(x) for x in wt]}

    @torch.no_grad()
    def mean_residual(self, X_u8, bs=32):
        dev = self.c.device
        tot, n = 0.0, 0
        for i in range(0, X_u8.shape[0], bs):
            I = X_u8[i:i + bs].to(dev).float().div(255).unsqueeze(1)
            tot += float(self.residual(I).sum())
            n += I.shape[0]
        return tot / max(n, 1)

    def state(self):
        return {"p": list(self.p), "k": self.k, "K": self.K, "lo": self.lo, "hi": self.hi, "mask_lo": self.mask_lo}

    def load(self, s):
        self.p = tuple(s["p"])
''')
Path("a1_leakage.py").write_text(r'''# a1_leakage.py - A1 HARD GATE: patient-level leakage + duplicate audit of the exact train/test split used by
# prepare.py (CBIS-DDSM official calc+mass train / test case-description CSVs, full mammograms at 256 px).
# PRE-STATED RULE:
#   FAIL  if any patient appears in both train and test, OR any train/test image pair is a (near-)duplicate
#         (Pearson correlation of 64x64 thumbnails > 0.995).
#   WARN  (not a leakage failure) for duplicates / label conflicts INSIDE a split; they are reported with counts.
import os, sys, json, subprocess
import numpy as np, pandas as pd
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_cache

TAG = "A1"
hit = subprocess.run("find /kaggle/input -maxdepth 6 -name dicom_info.csv", shell=True,
                     capture_output=True, text=True).stdout.split()
if not hit:
    raise SystemExit("CBIS-DDSM not attached")
RAW = os.path.dirname(os.path.dirname(hit[0]))
info = pd.read_csv(os.path.join(RAW, "csv/dicom_info.csv"))
full = info[info["SeriesDescription"] == "full mammogram images"]
lookup = full.set_index("PatientName")["image_path"].to_dict()

def col(df, *names):
    low = {c.lower().strip(): c for c in df.columns}
    for n in names:
        if n in low:
            return low[n]
    raise KeyError(f"none of {names} in columns {list(df.columns)[:12]}")

rows = []
for split, csvs in [("train", ["csv/calc_case_description_train_set.csv", "csv/mass_case_description_train_set.csv"]),
                    ("test", ["csv/calc_case_description_test_set.csv", "csv/mass_case_description_test_set.csv"])]:
    for c in csvs:
        df = pd.read_csv(os.path.join(RAW, c))
        pid, path, pat = col(df, "patient_id"), col(df, "image file path"), col(df, "pathology")
        side, view = col(df, "left or right breast"), col(df, "image view")
        for _, r in df.iterrows():
            name = str(r[path]).split("/")[0]
            rel = lookup.get(name)
            ok = rel is not None and os.path.exists(os.path.join(RAW, "jpeg", rel.replace("CBIS-DDSM/jpeg/", "")))
            rows.append({"split": split, "csv": os.path.basename(c), "patient": str(r[pid]), "side": str(r[side]),
                         "view": str(r[view]), "name": name, "jpeg": rel if ok else None,
                         "label": 1 if r[pat] == "MALIGNANT" else 0})
R = pd.DataFrame(rows)
R_ok = R[R.jpeg.notna()]
rep = {"rows_total": len(R), "rows_with_existing_jpeg": len(R_ok)}

# ---- 1. patient overlap train/test (true patient id, e.g. P_00001) ----
ptr, pte = set(R_ok[R_ok.split == "train"].patient), set(R_ok[R_ok.split == "test"].patient)
ov = sorted(ptr & pte)
rep["patients_train"], rep["patients_test"], rep["patients_in_both"] = len(ptr), len(pte), len(ov)
rep["patients_in_both_examples"] = ov[:10]
# ---- 2. same image (same jpeg) in both splits ----
itr, ite = set(R_ok[R_ok.split == "train"].jpeg), set(R_ok[R_ok.split == "test"].jpeg)
rep["images_in_both_splits"] = len(itr & ite)
# ---- 3. duplicates / label conflicts inside each split (one CSV row per abnormality) ----
for s in ("train", "test"):
    g = R_ok[R_ok.split == s].groupby("jpeg")
    n_rows, n_img = int((R_ok.split == s).sum()), int(g.ngroups)
    conflict = int((g.label.nunique() > 1).sum())
    rep[f"{s}_rows"], rep[f"{s}_unique_images"] = n_rows, n_img
    rep[f"{s}_rows_that_repeat_an_image"] = n_rows - n_img
    rep[f"{s}_images_with_conflicting_labels"] = conflict
    rep[f"{s}_breasts_per_patient_median"] = float(R_ok[R_ok.split == s].groupby("patient").side.nunique().median())

# ---- 4. near-duplicates in the ACTUAL caches used for training/testing ----
def thumbs(x):
    t = x.reshape(x.shape[0], 64, 4, 64, 4).mean((2, 4)).reshape(x.shape[0], -1).astype(np.float64)
    t -= t.mean(1, keepdims=True)
    t /= np.linalg.norm(t, axis=1, keepdims=True) + 1e-9
    return t
xtr, ytr = load_cache("train"); xte, yte = load_cache("test")
A, B = thumbs(xtr), thumbs(xte)
C = B @ A.T                                                   # (n_test, n_train) correlations
best = C.max(1)
rep["cache_train_n"], rep["cache_test_n"] = int(len(xtr)), int(len(xte))
rep["test_to_train_max_corr_quantiles"] = {q: float(np.quantile(best, q)) for q in (0.5, 0.9, 0.99, 1.0)}
rep["test_train_pairs_corr_gt_0.995"] = int((C > 0.995).sum())
rep["test_train_pairs_exact_equal"] = int(sum(int(np.any(np.all(xtr.reshape(len(xtr), -1) == xte[i].reshape(1, -1), 1)))
                                              for i in np.where(best > 0.999)[0]))
CT = A @ A.T; np.fill_diagonal(CT, -1)
rep["train_cache_pairs_corr_gt_0.995"] = int((np.triu(CT, 1) > 0.995).sum())
same_lab = (ytr[:, None] == ytr[None, :])
rep["train_cache_dup_pairs_with_different_labels"] = int(((np.triu(CT, 1) > 0.995) & ~same_lab).sum())

leak = rep["patients_in_both"] > 0 or rep["images_in_both_splits"] > 0 or rep["test_train_pairs_corr_gt_0.995"] > 0
rep["verdict"] = "FAIL (leakage)" if leak else "PASS (no patient/image overlap between train and test)"
rep["warnings"] = []
if rep["train_rows_that_repeat_an_image"] > 0:
    rep["warnings"].append(f"train: {rep['train_rows_that_repeat_an_image']} CSV rows repeat an image "
                           f"(prepare.py adds one copy per abnormality row)")
if rep["train_images_with_conflicting_labels"] > 0:
    rep["warnings"].append(f"train: {rep['train_images_with_conflicting_labels']} images carry BOTH labels")
if rep["train_cache_dup_pairs_with_different_labels"] > 0:
    rep["warnings"].append(f"train cache: {rep['train_cache_dup_pairs_with_different_labels']} duplicate pairs "
                           f"with different labels")
save_json(rep, "A1_leakage.json")
for k, v in rep.items():
    log(TAG, f"{k}: {v}")
''')
Path("a2_jensen.py").write_text(r'''# a2_jensen.py - A2: conditional-mean ("Jensen") test on the REAL trained network.
# For real test images x0 and noise levels t: x_t -> (i) x0_hat = the network's one-step estimate (what the
# Experiment 2-5 physics loss was applied to), (ii) Bastek-style sample estimate (deterministic DDIM jump to t=0, then
# x0_hat there = the "sample estimate" of Bastek et al., ICLR 2025, App. A.4: one DDIM step t -> least-noisy step, then
# x0_hat; two network passes), (iii) K stochastic samples of the network's OWN reverse process started at x_t (exact DDPM ancestral
# sampling, conditional label, no guidance). Statistics on all of them, ladder from linear to strongly nonlinear:
#   L1 mean intensity (breast) | L2 high-pass variance | L3 log L2 | L4 the training physics residual (PhysicsNLF)
# PRE-STATED t* RULE: band = 10-90 % quantiles of L4 on ALL real test images; f_out(t) = share of x0_hat outside the
#   band; t* = the largest grid t such that f_out(t') <= f_out(real) + 0.10 for every grid t' <= t (t* = 0 if t=5 fails).
import os, sys, json, time, argparse
import numpy as np, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_cache, to_model_domain, ctx_of, eps_pred, load_model, main_ckpt, \
    optional_25_ckpt
from pinn_physics import PhysicsNLF

ap = argparse.ArgumentParser()
ap.add_argument("--which", default="main")            # main | 25
ap.add_argument("--n_img", type=int, default=16)
ap.add_argument("--k", type=int, default=4)              # samples per image for t <= 100
ap.add_argument("--k_hi", type=int, default=2)           # samples per image for t > 100 (exact sampling costs t passes)
ap.add_argument("--grid", default="5,10,20,50,100,200,300,500")
a = ap.parse_args()
TAG = f"A2-{a.which}"
dev = torch.device("cuda")
path = main_ckpt() if a.which == "main" else optional_25_ckpt()
if path is None:
    log(TAG, "no 25 %-data checkpoint found -> skipped"); raise SystemExit(0)
unet, ac, meta = load_model(path, dev)
log(TAG, f"checkpoint {path} | epoch {meta.get('epoch')}")
nlf = PhysicsNLF(dev)
if meta["nlf"] is not None:
    nlf.load(meta["nlf"])
else:
    raise SystemExit("snapshot has no NLF state")
betas = 1 - ac / torch.cat([torch.ones(1, device=dev, dtype=ac.dtype), ac[:-1]])

def ladder(x_pm1):
    I = ((x_pm1 + 1) / 2).clamp(0, 1).float()
    h = I - nlf.blur(I)
    mu = torch.nn.functional.avg_pool2d(I, 7, 1, 3, count_include_pad=False)
    m = (mu > 0.04).float()
    n = m.sum((1, 2, 3)).clamp_min(1)
    L1 = (I * m).sum((1, 2, 3)) / n
    L2 = (h * h * m).sum((1, 2, 3)) / n
    L3 = torch.log(L2 + 1e-12)
    L4 = nlf.residual(I)
    return torch.stack([L1, L2, L3, L4], 1).cpu().numpy()

@torch.no_grad()
def x0hat(x, t, ctx):
    e = eps_pred(unet, x, t, ctx, amp=True)
    ab = ac[t].float()
    return ((x - (1 - ab).sqrt() * e) / ab.sqrt()).clamp(-1, 1), e

@torch.no_grad()
def ancestral(x, t, ctx, gen):
    """exact DDPM reverse process (FIXED_SMALL variance, clip like the MONAI scheduler) from step t down to 0"""
    for s in range(t, 0, -1):
        x0, _ = x0hat(x, s, ctx)
        ab, abp, b = ac[s].float(), ac[s - 1].float(), betas[s].float()
        mean = (abp.sqrt() * b / (1 - ab)) * x0 + ((1 - b).sqrt() * (1 - abp) / (1 - ab)) * x
        var = ((1 - abp) / (1 - ab) * b).clamp_min(1e-20)
        x = mean + var.sqrt() * torch.randn(x.shape, generator=gen, device=dev)
    return x0hat(x, 0, ctx)[0]

xte, yte = load_cache("test")
real_all = np.concatenate([ladder(to_model_domain(xte[i:i + 32], dev)) for i in range(0, len(xte), 32)])
lo, hi = np.quantile(real_all[:, 3], [0.1, 0.9]); sd = real_all.std(0); mu_r = real_all.mean(0)
f_real = float(np.mean((real_all[:, 3] < lo) | (real_all[:, 3] > hi)))
r = np.random.default_rng(0)
idx = np.concatenate([r.choice(np.where(yte == c)[0], a.n_img // 2, replace=False) for c in (0, 1)])
X0 = to_model_domain(xte[idx], dev); lab = yte[idx]
grid = [int(g) for g in a.grid.split(",")]
gen = torch.Generator(device=dev).manual_seed(123)
rows = []
t0 = time.time()
for t in grid:
    ab = ac[t].float()
    eps = torch.randn(X0.shape, generator=gen, device=dev)
    xt = ab.sqrt() * X0 + (1 - ab).sqrt() * eps
    ctx = ctx_of(lab, dev)
    m_hat, e_hat = x0hat(xt, t, ctx)
    # Bastek sample estimate: deterministic DDIM jump x_t -> step 0, then x0_hat at step 0
    x_0 = ac[0].float().sqrt() * m_hat + (1 - ac[0].float()).sqrt() * e_hat
    bast = x0hat(x_0, 0, ctx)[0]
    K = a.k if t <= 100 else a.k_hi
    samples = []
    for k in range(K):
        samples.append(ancestral(xt.clone(), t, ctx, gen))
    S = torch.stack(samples, 1)                                  # (N, K, 1, H, W)
    Ls = np.stack([ladder(S[:, k]) for k in range(K)], 1)        # (N, K, 4)
    Lm, Lb, Lr = ladder(m_hat), ladder(bast), ladder(X0)
    spread = float(((S - S.mean(1, keepdim=True)) ** 2).mean())  # posterior variance per pixel (empirical)
    gapmean = float(((m_hat - S.mean(1)) ** 2).mean())
    out = lambda L4: float(np.mean((L4 < lo) | (L4 > hi)))
    rows.append({"t": t, "abar": float(ab), "k_samples": K,
                 "gap_sd_xhat": ((Lm.mean(0) - mu_r) / sd).tolist(),
                 "gap_sd_bastek": ((Lb.mean(0) - mu_r) / sd).tolist(),
                 "gap_sd_samples": ((Ls.reshape(-1, 4).mean(0) - mu_r) / sd).tolist(),
                 "gap_sd_real_subset": ((Lr.mean(0) - mu_r) / sd).tolist(),
                 "jensen_S_of_mean_minus_mean_of_S_sd": ((Lm - Ls.mean(1)).mean(0) / sd).tolist(),
                 "out_band_xhat": out(Lm[:, 3]), "out_band_bastek": out(Lb[:, 3]),
                 "out_band_samples": out(Ls[:, :, 3].ravel()), "out_band_real_subset": out(Lr[:, 3]),
                 "posterior_var_per_px": spread, "sq_dist_xhat_to_sample_mean_per_px": gapmean})
    log(TAG, f"t={t:4d} out-of-band x0_hat {rows[-1]['out_band_xhat']:.2f} | samples {rows[-1]['out_band_samples']:.2f} | "
             f"bastek {rows[-1]['out_band_bastek']:.2f} | L4 gap (SD) x0_hat {rows[-1]['gap_sd_xhat'][3]:+.2f} "
             f"samples {rows[-1]['gap_sd_samples'][3]:+.2f} | {(time.time() - t0) / 60:.1f} min")
thr = f_real + 0.10
t_star = 0
for row in rows:
    if row["out_band_xhat"] <= thr:
        t_star = row["t"]
    else:
        break
res = {"checkpoint": path, "epoch": meta.get("epoch"), "n_img": int(a.n_img), "k_samples": [a.k, a.k_hi],
       "real_band_L4": [float(lo), float(hi)], "f_out_real_all": f_real, "threshold": thr, "t_star": t_star,
       "real_mean": mu_r.tolist(), "real_sd": sd.tolist(), "rows": rows}
save_json(res, f"A2_jensen_{a.which}.json")
log(TAG, f"t* = {t_star} (rule: x0_hat out-of-band share <= {thr:.2f} for every grid t up to t*)")
''')
Path("a3_texture.py").write_text(r'''# a3_texture.py - A3 (minimal, CPU): is the measured "noise curve" noise or tissue texture, and how much detail
# survives at 256 x 256?
# PRE-STATED RULES:
#   resolution test: halving the resolution of pure white-like noise divides its high-pass variance by ~4.
#     ratio V(half res)/V(full res) in FLAT breast regions <= 0.35 -> "noise-like";  >= 0.70 -> "texture-dominated";
#     otherwise "mixed".
#   calibration: noise with a KNOWN curve is added to heavily smoothed real images; the estimator must recover it
#     up to the analytically known attenuation of the 7-tap high-pass filter (reported as recovered/expected).
import os, sys, json, subprocess
import numpy as np, pandas as pd, torch
from PIL import Image
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_cache
from pinn_physics import PhysicsNLF

TAG = "A3"
rng = np.random.default_rng(0)
hit = subprocess.run("find /kaggle/input -maxdepth 6 -name dicom_info.csv", shell=True,
                     capture_output=True, text=True).stdout.split()
RAW = os.path.dirname(os.path.dirname(hit[0]))
info = pd.read_csv(os.path.join(RAW, "csv/dicom_info.csv"))
jp = lambda rel: os.path.join(RAW, "jpeg", rel.replace("CBIS-DDSM/jpeg/", ""))
full = info[info["SeriesDescription"] == "full mammogram images"]
paths = [jp(p) for p in full["image_path"].tolist() if os.path.exists(jp(p))]
rep = {}
nlf = PhysicsNLF(torch.device("cpu"))

# ---- (a) native resolution of the JPEG files actually used ----
sel = rng.choice(len(paths), min(60, len(paths)), replace=False)
sizes = []
for i in sel:
    with Image.open(paths[i]) as im:
        sizes.append(im.size)                                         # (W, H)
W = np.array([s[0] for s in sizes]); H = np.array([s[1] for s in sizes])
rep["native_size_median_WxH"] = [int(np.median(W)), int(np.median(H))]
rep["downsample_factor_to_256_median"] = [float(np.median(W) / 256), float(np.median(H) / 256)]
rep["jpeg_mode_examples"] = list({Image.open(paths[i]).mode for i in sel[:10]})

# ---- (b) resolution test in flat and all breast regions ----
def hp_var(img01, flat_q=0.3):
    I = torch.from_numpy(img01).float()[None, None]
    h = I - nlf.blur(I)
    mu = torch.nn.functional.avg_pool2d(I, 7, 1, 3, count_include_pad=False)
    gy, gx = torch.gradient(nlf.blur(I)[0, 0])
    gm = (gx ** 2 + gy ** 2).sqrt()
    breast = (mu[0, 0] > 0.04)
    if breast.sum() < 100:
        return None
    gb = gm[breast].numpy()
    thr = float(np.quantile(gb[np.random.default_rng(1).choice(len(gb), min(len(gb), 2_000_000), replace=False)], flat_q))
    flat = breast & (gm <= thr)
    hh = h[0, 0] ** 2
    return float(hh[flat].mean()), float(hh[breast].mean())

ratios = {"flat_512_over_1024": [], "flat_256_over_512": [], "all_512_over_1024": [], "all_256_over_512": []}
for i in sel[:40]:
    g = Image.open(paths[i]).convert("L")
    v = {}
    for r in (1024, 512, 256):
        a = np.asarray(g.resize((r, r)), dtype=np.float32)          # same PIL default resize as prepare.py
        a = (a - a.min()) / (a.max() - a.min() + 1e-8)
        v[r] = hp_var(a)
    if any(x is None for x in v.values()):
        continue
    ratios["flat_512_over_1024"].append(v[512][0] / v[1024][0]); ratios["flat_256_over_512"].append(v[256][0] / v[512][0])
    ratios["all_512_over_1024"].append(v[512][1] / v[1024][1]); ratios["all_256_over_512"].append(v[256][1] / v[512][1])
rep["resolution_ratios_median"] = {k: float(np.median(x)) for k, x in ratios.items() if x}
rf = rep["resolution_ratios_median"].get("flat_256_over_512", np.nan)
rep["resolution_verdict"] = ("noise-like" if rf <= 0.35 else "texture-dominated" if rf >= 0.70 else "mixed") \
    if np.isfinite(rf) else "not computed"

# ---- (c) region test at 256: binned high-pass variance, flat vs edge pixels ----
x, y = load_cache("train")
idx = rng.choice(len(x), min(300, len(x)), replace=False)
I = torch.from_numpy(x[idx]).float().unsqueeze(1)
mn = I.amin(dim=(1, 2, 3), keepdim=True); mx = I.amax(dim=(1, 2, 3), keepdim=True)
I = (I - mn) / (mx - mn + 1e-8)
h = I - nlf.blur(I)
mu = torch.nn.functional.avg_pool2d(I, 7, 1, 3, count_include_pad=False)
v = torch.nn.functional.avg_pool2d(h * h, 7, 1, 3, count_include_pad=False)
B = nlf.blur(I)
gx = B[..., :, 2:] - B[..., :, :-2]; gy = B[..., 2:, :] - B[..., :-2, :]
gm = torch.zeros_like(I); gm[..., 1:-1, 1:-1] = (gx[..., 1:-1, :] ** 2 + gy[..., :, 1:-1] ** 2).sqrt()
breast = mu > 0.04
_g = gm[breast].numpy(); _g = _g[rng.choice(len(_g), min(len(_g), 2_000_000), replace=False)]
thr_lo, thr_hi = float(np.quantile(_g, 0.3)), float(np.quantile(_g, 0.7))
edges = np.linspace(0.08, 0.95, 17)
curve = {"flat": [], "edge": [], "all": []}
for name, m in (("flat", breast & (gm <= thr_lo)), ("edge", breast & (gm >= thr_hi)), ("all", breast)):
    mu_m, v_m = mu[m].numpy(), v[m].numpy()
    b = np.digitize(mu_m, edges) - 1
    curve[name] = [float(v_m[b == k].mean()) if np.sum(b == k) > 50 else None for k in range(16)]
rep["region_curves_bin_edges"] = edges.tolist()
rep["region_curves"] = curve
ratio = [e / f for e, f in zip(curve["edge"], curve["flat"]) if e and f]
rep["edge_over_flat_variance_median"] = float(np.median(ratio)) if ratio else None

# ---- (d) synthetic calibration: known noise curve on smoothed real backgrounds ----
b_, a_, c_ = 1.784e-3, -4.218e-3, 3.308e-3                       # the real Experiment-2 fit
g = nlf.g.numpy(); g2 = np.outer(g, g); delta = np.zeros((7, 7)); delta[3, 3] = 1
atten = float(((delta - g2) ** 2).sum())                          # fraction of white-noise variance kept by I - blur(I)
bg = torch.nn.functional.avg_pool2d(I[:50], 9, 1, 4, count_include_pad=False)
bg = nlf.blur(nlf.blur(bg))                                        # heavily smoothed "clean" background
sig2 = (b_ + a_ * bg + c_ * bg ** 2).clamp_min(1e-7)
noisy = bg + torch.randn(bg.shape, generator=torch.Generator().manual_seed(0)) * sig2.sqrt()
vb_clean, m_clean = nlf.stats(bg)
vb_noisy, m_noisy = nlf.stats(noisy)
cent = nlf.c.numpy(); known = b_ + a_ * cent + c_ * cent ** 2
rec = (vb_noisy * m_noisy).sum(0) / m_noisy.sum(0).clamp_min(1e-6)
cle = (vb_clean * m_clean).sum(0) / m_clean.sum(0).clamp_min(1e-6)
ok = (m_noisy.sum(0) > 1000).numpy()
rep["calibration_highpass_attenuation_expected"] = atten
rep["calibration_recovered_over_known_median"] = float(np.median((rec.numpy() / known)[ok]))
rep["calibration_recovered_over_expected_median"] = float(np.median((rec.numpy() / (known * atten))[ok]))
rep["calibration_texture_leak_clean_over_known_median"] = float(np.median((cle.numpy() / known)[ok]))

# ---- (e) lesion size at 256 x 256 from ROI masks ----
try:
    masks = info[info["SeriesDescription"] == "ROI mask images"].set_index("PatientName")["image_path"].to_dict()
    les = []
    for c in ("csv/mass_case_description_train_set.csv", "csv/calc_case_description_train_set.csv"):
        df = pd.read_csv(os.path.join(RAW, c))
        mcol = [k for k in df.columns if k.lower().strip() == "roi mask file path"][0]
        typ = "mass" if "mass" in c else "calc"
        for p in df[mcol].sample(min(100, len(df)), random_state=0):
            rel = masks.get(str(p).split("/")[0])
            if rel is None or not os.path.exists(jp(rel)):
                continue
            m = np.asarray(Image.open(jp(rel)).convert("L"))
            if np.mean((m > 30) & (m < 225)) > 0.02:                 # not a binary mask (dataset mislabel) -> skip
                continue
            mb = m > 127
            if mb.sum() == 0:
                continue
            ys, xs = np.where(mb)
            Hn, Wn = m.shape
            les.append({"type": typ, "area_px_256": float(mb.mean() * 256 * 256),
                        "bbox_h_256": float((ys.max() - ys.min() + 1) * 256 / Hn),
                        "bbox_w_256": float((xs.max() - xs.min() + 1) * 256 / Wn)})
    L = pd.DataFrame(les)
    out = {}
    for typ, g_ in L.groupby("type"):
        d = 2 * np.sqrt(g_.area_px_256 / np.pi)
        out[typ] = {"n": int(len(g_)), "equiv_diameter_px_median": float(d.median()),
                    "equiv_diameter_px_q10_q90": [float(d.quantile(0.1)), float(d.quantile(0.9))],
                    "bbox_min_side_px_median": float(np.minimum(g_.bbox_h_256, g_.bbox_w_256).median()),
                    "fraction_diameter_below_5px": float((d < 5).mean())}
    rep["lesion_size_at_256"] = out
except Exception as e:
    rep["lesion_size_at_256"] = f"skipped: {repr(e)[:200]}"

save_json(rep, "A3_texture.json")
for k, v in rep.items():
    log(TAG, f"{k}: {v}")
''')
Path("a4_fp.py").write_text(r'''# a4_fp.py - A4: validation of the Fokker-Planck (score-FPE) residual estimator on the REAL trained network.
# No training. Network: Experiment-2 full-data baseline (epoch 40). Score s(x,t) = -eps_hat(x,t) / sqrt(1 - abar_t).
# Score FPE of the VP-SDE (tau = t/1000, beta(tau) = -d log abar / d tau):
#     d_tau s = 1/2 beta grad_x( x.s + div s + |s|^2 )
# Projected on a unit direction u (no symmetry of J_s needed):
#     r(u) = u.d_tau s - 1/2 beta [ u.s + (J_s u).(x + 2 s) + d_u(div s) ],   d_u(div s) ~ v.(d_u J_s) v  (Hutchinson)
# Estimators compared (same u, same Rademacher v):
#   REF  : exact directional derivatives by forward-mode AD (torch.func.jvp, nested for d_u div s), fp32
#          (fallback if jvp is not supported by the network: central finite differences in fp64, p = 1e-4)
#   FD   : central finite differences with per-pixel step p (h_u = 256 p for unit u, h_v = p for Rademacher v),
#          in fp32 and in fp16 autocast (= training precision), p in {1e-3, 3e-3, 1e-2}
#   d_tau s always by central differences in the integer timestep (+-k steps, k in {1, 2, 5} reported)
#   product estimator r(u,v1) r(u,v2) (independent v) = unbiased estimate of (u.R)^2
#   guided score (CFG scale 4, as used for sampling): FD fp32, p = 3e-3, 2 probes
# PRE-STATED RULES:
#   estimator usable at precision P : for the best p, rel.error = RMS(r_P - r_REF)/RMS(r_REF) <= 0.10 at every grid
#                                     t in [10, 300]
#   FP residual measurable at t      : z = mean(product)/SE(product) > 3 (REF, all images x directions)
import os, sys, math, time, copy, argparse
import numpy as np, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_cache, to_model_domain, ctx_of, eps_pred, load_model, main_ckpt, OUT

ap = argparse.ArgumentParser()
ap.add_argument("--n_img", type=int, default=6)
ap.add_argument("--n_u", type=int, default=3)
ap.add_argument("--n_v", type=int, default=4)
ap.add_argument("--grid", default="10,20,50,100,200,300,500,700")
ap.add_argument("--ps", default="1e-3,3e-3,1e-2")
ap.add_argument("--dts", default="1,2,5")
ap.add_argument("--gs", type=float, default=4.0)
ap.add_argument("--p_guided", type=float, default=3e-3)
ap.add_argument("--fp64_check_t", default="20,100")
ap.add_argument("--fp64_p", type=float, default=1e-4)
a = ap.parse_args()
TAG = "A4"
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
dev = torch.device("cuda")
path = main_ckpt()
unet, ac, meta = load_model(path, dev)                      # fp32 weights (fp16 mode = autocast, as in training)
log(TAG, f"checkpoint {path} | epoch {meta.get('epoch')}")
T = ac.numel()
DTAU = 1.0 / T
LOGAB = torch.log(ac)
GRID = [int(g) for g in a.grid.split(",")]
PS = [float(p) for p in a.ps.split(",")]
DTS = [int(k) for k in a.dts.split(",")]
assert max(GRID) + max(DTS) < T and min(GRID) - max(DTS) >= 0
NETS = {"fp32": unet, "fp16": unet}
CHUNK = {"fp16": 8, "fp32": 6, "fp64": 2}


def beta_eff(t):
    return float(-(LOGAB[t + 1] - LOGAB[t - 1]) / (2 * DTAU))


def sig(t):
    return math.sqrt(1.0 - float(ac[t]))


def dot(p, q):
    return (p * q).flatten(1).sum(1)


def S(xb, t, lab, mode, gs=None, chunk=None):
    """score of the network at integer step t; mode fp32 | fp16 (autocast) | fp64 (double copy)"""
    net = NETS[mode]
    dt = torch.float64 if mode == "fp64" else torch.float32
    xb = xb.to(dt)
    B = xb.shape[0]
    ch = chunk or CHUNK[mode]
    e = eps_pred(net, xb, t, ctx_of([lab] * B, dev, dt), amp=(mode == "fp16"), chunk=ch).to(dt)
    if gs is not None:
        eu = eps_pred(net, xb, t, ctx_of([-1] * B, dev, dt), amp=(mode == "fp16"), chunk=ch).to(dt)
        e = eu + gs * (e - eu)
    return -e / sig(t)


@torch.no_grad()
def lhs(x, t, lab, mode, U, k=1, gs=None):
    d = (S(x, t + k, lab, mode, gs) - S(x, t - k, lab, mode, gs)) / (2 * k * DTAU)
    return dot(U.to(d.dtype), d.expand(U.shape[0], -1, -1, -1))


@torch.no_grad()
def terms_fd(x, t, lab, U, V, p, mode, gs=None, k=1):
    dt = torch.float64 if mode == "fp64" else torch.float32
    x, U, V = x.to(dt), U.to(dt), V.to(dt)
    nu, nv = U.shape[0], V.shape[0]
    hu, hv = 256.0 * p, p
    s0 = S(x, t, lab, mode, gs)
    L = lhs(x, t, lab, mode, U, k, gs)
    us = dot(U, s0.expand(nu, -1, -1, -1))
    Xb = x.expand(nu, -1, -1, -1)
    sj = S(torch.cat([Xb + hu * U, Xb - hu * U]), t, lab, mode, gs)
    Ju = (sj[:nu] - sj[nu:]) / (2 * hu)
    Jt = dot(Ju, (x + 2 * s0).expand(nu, -1, -1, -1))
    Up = U[:, None].expand(nu, nv, *U.shape[1:]).reshape(nu * nv, *U.shape[1:])
    Vp = V[None].expand(nu, nv, *V.shape[1:]).reshape(nu * nv, *V.shape[1:])
    Xm = x.expand(nu * nv, -1, -1, -1)
    n = nu * nv
    D2 = torch.zeros(n, dtype=dt, device=dev)
    for sgn_u, sgn_v, c in ((1, 1, 1), (1, -1, -1), (-1, 1, -1), (-1, -1, 1)):    # mixed second difference
        D2 += c * dot(Vp, S(Xm + sgn_u * hu * Up + sgn_v * hv * Vp, t, lab, mode, gs))
    D2 = (D2 / (4 * hu * hv)).view(nu, nv)
    return {"L": L, "us": us, "Jt": Jt, "D2": D2, "beta": beta_eff(t)}


def terms_jvp(x, t, lab, U, V, k=1):
    from torch.func import jvp
    nu, nv = U.shape[0], V.shape[0]
    with torch.no_grad():
        s0 = S(x, t, lab, "fp32")
        L = lhs(x, t, lab, "fp32", U, k)
    us = dot(U, s0.expand(nu, -1, -1, -1))
    Xb = x.expand(nu, -1, -1, -1).contiguous()
    _, Ju = jvp(lambda z: S(z, t, lab, "fp32", chunk=3), (Xb,), (U.contiguous(),))
    Jt = dot(Ju, (x + 2 * s0).expand(nu, -1, -1, -1))
    D2 = torch.zeros(nu, nv, device=dev)
    for j in range(nv):
        vj = V[j:j + 1].expand(nu, -1, -1, -1).contiguous()

        def g(z, vj=vj):
            _, jv = jvp(lambda w: S(w, t, lab, "fp32", chunk=1), (z,), (vj,))
            return dot(vj, jv)

        _, d = jvp(g, (Xb,), (U.contiguous(),))
        D2[:, j] = d.detach()
    return {"L": L, "us": us.detach(), "Jt": Jt.detach(), "D2": D2, "beta": beta_eff(t)}


def resid(d):
    return (d["L"][:, None] - 0.5 * d["beta"] * (d["us"][:, None] + d["Jt"][:, None] + d["D2"])).double().cpu().numpy()


def np_terms(d):
    b = 0.5 * d["beta"]
    f = lambda z: z.double().cpu().numpy()
    return {"L": f(d["L"]), "us": b * f(d["us"]), "Jt": b * f(d["Jt"]), "D2": b * f(d["D2"])}


def rel(p, q):
    return float(np.sqrt(np.mean((p - q) ** 2)) / (np.sqrt(np.mean(q ** 2)) + 1e-300))


def prod_stats(r):
    """r: (N, n_u, n_v) -> product (U-statistic over probe pairs) = unbiased (u.R)^2 per (image, u)"""
    V_ = r.shape[-1]
    s1, s2 = r.sum(-1), (r ** 2).sum(-1)
    P = ((s1 ** 2 - s2) / (V_ * (V_ - 1))).ravel()
    m, se = float(P.mean()), float(P.std(ddof=1) / np.sqrt(len(P)))
    return {"mean_sq_residual": m, "se": se, "z": m / (se + 1e-300), "rms_residual": float(np.sqrt(max(m, 0.0))),
            "single_probe_var": float(r.var(-1, ddof=1).mean())}


# ---------------- data: n_img real test images (half benign / half malignant), fixed noise ----------------
xte, yte = load_cache("test")
rng = np.random.default_rng(0)
idx = np.concatenate([rng.choice(np.where(yte == c)[0], a.n_img // 2, replace=False) for c in (0, 1)])
X0 = to_model_domain(xte[idx], dev)
LAB = [int(v) for v in yte[idx]]
gen = torch.Generator(device=dev).manual_seed(7)
H, W = X0.shape[-2:]
U = torch.randn((a.n_u, 1, H, W), generator=gen, device=dev)
U = U / U.flatten(1).norm(dim=1).view(-1, 1, 1, 1)                        # unit directions (per-pixel RMS 1/256)
V = (torch.randint(0, 2, (a.n_v, 1, H, W), generator=gen, device=dev) * 2 - 1).float()   # Rademacher probes
NOISE = torch.randn(X0.shape, generator=gen, device=dev)

# ---------------- reference method: JVP (fallback FD fp64) ----------------
REF = "jvp"
try:
    _ = terms_jvp(X0[:1] * 0 + 0.1, GRID[0], LAB[0], U, V[:1])        # full direction batch -> also a memory test
    log(TAG, "forward-mode AD (torch.func.jvp, nested) works on this network -> reference = JVP fp32 | peak "
             f"{torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
except Exception as e:
    REF = "fd64"
    log(TAG, f"jvp NOT usable ({repr(e)[:200]}) -> reference = central FD in fp64, p = {a.fp64_p}, on 2 images")
    NETS["fp64"] = copy.deepcopy(unet).double()
    keep = [0, X0.shape[0] - 1]                                           # one benign, one malignant
    X0, NOISE, LAB = X0[keep], NOISE[keep], [LAB[k] for k in keep]
torch.cuda.empty_cache()

methods = [(m, p) for m in ("fp32", "fp16") for p in PS]
rows = []
raw = {}
t_all = time.time()
for t in GRID:
    t1 = time.time()
    ab = float(ac[t])
    Xt = math.sqrt(ab) * X0 + math.sqrt(1 - ab) * NOISE
    R_ref, R_m, terms_ref, terms_m, dts_rows, R_g, R_ug = [], {mp: [] for mp in methods}, [], {mp: [] for mp in methods}, [], [], []
    for i in range(Xt.shape[0]):
        x, lab = Xt[i:i + 1], LAB[i]
        d_ref = terms_jvp(x, t, lab, U, V) if REF == "jvp" else terms_fd(x, t, lab, U, V, a.fp64_p, "fp64")
        R_ref.append(resid(d_ref)); terms_ref.append(np_terms(d_ref))
        # time-derivative step sensitivity (reference spatial terms, fp32 time differences)
        Lk = {k: lhs(x, t, lab, "fp32", U, k).double().cpu().numpy() for k in DTS}
        dts_rows.append(Lk)
        for mp in methods:
            d = terms_fd(x, t, lab, U, V, mp[1], mp[0])
            R_m[mp].append(resid(d)); terms_m[mp].append(np_terms(d))
        dg = terms_fd(x, t, lab, U, V[:2], a.p_guided, "fp32", gs=a.gs)
        du = terms_fd(x, t, lab, U, V[:2], a.p_guided, "fp32")
        R_g.append(resid(dg)); R_ug.append(resid(du))
    R_ref = np.stack(R_ref)
    st = prod_stats(R_ref)
    TR = {k: np.stack([d[k] for d in terms_ref]) for k in ("L", "us", "Jt", "D2")}
    lhs_rms = float(np.sqrt(np.mean(TR["L"] ** 2)))
    row = {"t": t, "abar": ab, "beta_eff": beta_eff(t), "reference": REF, **st,
           "lhs_rms": lhs_rms, "relative_residual_rho": st["rms_residual"] / (lhs_rms + 1e-300),
           "term_rms": {k: float(np.sqrt(np.mean(v ** 2))) for k, v in TR.items()},
           "dtau_step_rel_change_vs_k1": {}, "methods": {}}
    L1 = np.stack([d[DTS[0]] for d in dts_rows])
    for k in DTS[1:]:
        row["dtau_step_rel_change_vs_k1"][str(k)] = rel(np.stack([d[k] for d in dts_rows]), L1)
    for mp in methods:
        Rm = np.stack(R_m[mp])
        TM = {k: np.stack([d[k] for d in terms_m[mp]]) for k in ("L", "us", "Jt", "D2")}
        row["methods"][f"{mp[0]}_p{mp[1]:g}"] = {
            "rel_err_r": rel(Rm, R_ref), "rel_err_Jterm": rel(TM["Jt"], TR["Jt"]), "rel_err_D2term": rel(TM["D2"], TR["D2"]),
            "rel_err_lhs": rel(TM["L"], TR["L"]), "z_product": prod_stats(Rm)["z"]}
        raw[f"r_{mp[0]}_p{mp[1]:g}_t{t}"] = Rm
    Rg, Rug = np.stack(R_g), np.stack(R_ug)
    Pg, Pu = (Rg[..., 0] * Rg[..., 1]).ravel(), (Rug[..., 0] * Rug[..., 1]).ravel()
    row["guided"] = {"gs": a.gs, "p": a.p_guided, "mean_sq_residual_guided": float(Pg.mean()),
                     "mean_sq_residual_unguided_same_setting": float(Pu.mean()),
                     "rms_ratio_guided_over_unguided": float(np.sqrt(max(Pg.mean(), 0)) / (np.sqrt(max(Pu.mean(), 0)) + 1e-300)),
                     "z_guided": float(Pg.mean() / (Pg.std(ddof=1) / np.sqrt(len(Pg)) + 1e-300))}
    raw[f"r_ref_t{t}"] = R_ref
    rows.append(row)
    save_json({"partial": True, "reference": REF, "rows": rows}, "A4_fp_partial.json")
    best16 = min((v["rel_err_r"], k) for k, v in row["methods"].items() if k.startswith("fp16"))
    best32 = min((v["rel_err_r"], k) for k, v in row["methods"].items() if k.startswith("fp32"))
    log(TAG, f"t={t:4d} | z={st['z']:6.2f} rho={row['relative_residual_rho']:.3f} | best fp32 {best32[1]} err {best32[0]:.3f}"
             f" | best fp16 {best16[1]} err {best16[0]:.3f} | guided/unguided RMS "
             f"{row['guided']['rms_ratio_guided_over_unguided']:.2f} | {time.time() - t1:.0f} s")

# ---------------- fp64 cross-check of the JVP reference (1 image, 1 direction, 2 probes) ----------------
check = []
if REF == "jvp":
    try:
        NETS["fp64"] = copy.deepcopy(unet).double()
        for t in [int(v) for v in a.fp64_check_t.split(",") if v]:
            ab = float(ac[t])
            x = math.sqrt(ab) * X0[:1] + math.sqrt(1 - ab) * NOISE[:1]
            rj = resid(terms_jvp(x, t, LAB[0], U[:1], V[:2]))
            rf = resid(terms_fd(x, t, LAB[0], U[:1], V[:2], a.fp64_p, "fp64"))
            check.append({"t": t, "rel_diff_jvp_vs_fd64": rel(rj, rf)})
            log(TAG, f"fp64 cross-check t={t}: rel diff JVP vs FD64 = {check[-1]['rel_diff_jvp_vs_fd64']:.2e}")
        del NETS["fp64"]
    except Exception as e:
        check.append({"error": repr(e)[:300]})

# ---------------- verdicts ----------------
inwin = [r for r in rows if 10 <= r["t"] <= 300]
def worst(key):
    return max(r["methods"][key]["rel_err_r"] for r in inwin)
choice = {}
for prec in ("fp16", "fp32"):
    keys = [k for k in rows[0]["methods"] if k.startswith(prec)]
    wk = {k: worst(k) for k in keys}
    kb = min(wk, key=wk.get)
    choice[prec] = {"best_setting": kb, "worst_rel_err_t10_300": wk[kb], "usable": bool(wk[kb] <= 0.10),
                    "all_settings_worst_rel_err": wk}
meas = [r["t"] for r in rows if r["z"] > 3]
res = {"checkpoint": path, "epoch": meta.get("epoch"), "reference": REF, "n_img": int(X0.shape[0]), "n_u": a.n_u,
       "n_v": a.n_v, "grid": GRID, "rows": rows, "fp64_crosscheck": check, "precision_choice": choice,
       "t_with_z_gt_3": meas,
       "fp_residual_measurable_all_t_10_300": bool(all(r["z"] > 3 for r in inwin)),
       "verdict_estimator": ("fp16 FD usable (" + choice["fp16"]["best_setting"] + ")") if choice["fp16"]["usable"] else
                            ("fp16 NOT usable; fp32 FD usable (" + choice["fp32"]["best_setting"] + ")") if choice["fp32"]["usable"] else
                            "neither fp16 nor fp32 FD within 10 % of the reference -> FD estimator not usable as is",
       "minutes": (time.time() - t_all) / 60}
save_json(res, "A4_fp.json")
np.savez_compressed(os.path.join(OUT, "A4_raw.npz"), **raw)
log(TAG, f"estimator verdict: {res['verdict_estimator']}")
log(TAG, f"FP residual measurable (z>3) at t = {meas} | all t in [10,300]: {res['fp_residual_measurable_all_t_10_300']}")
log(TAG, f"done in {res['minutes']:.1f} min")
''')
Path("a5_bench.py").write_text(r'''# a5_bench.py - A5: compute benchmark for the Fokker-Planck (FP) training term (no real training).
#  1. unit test: the "manual recompute" gradient of the FP product loss  L = r(u,v1) r(u,v2)  equals autograd (fp64,
#     tiny network). Manual recompute = all 13 network evaluations without graph -> analytic dL/ds_k -> recompute each
#     evaluation with graph and backpropagate (dL/ds_k . s_k): exact gradient, memory of ONE evaluation.
#  2. baseline: one training micro-step of the original recipe (batch 2, 0.7 MSE + 0.2 (1-MS-SSIM) + 0.05 HF, fp16
#     autocast, GradScaler, Adam every 4 micro-steps, clip 1.0) - time and peak memory
#  3. FP term on ONE image (13 evaluations: t+-k, s0, +-h_u u, 2 probes x 4 mixed) - fp16 and fp32, recompute in groups
#     of 1 or 2 evaluations; full-graph autograd attempt at batch 1 (expected to run out of memory)
#  4. cost ratio = (T_base + T_fp / k) / T_base for "FP on one image every k micro-steps", k in {1, 2, 4, 8}
#     PRE-STATED SEED RULE (k = 4 = once per optimizer step): ratio <= 2 -> 5 seeds; <= 3 -> 4 seeds; > 3 -> reconsider
#  5. Phase B warm start: does base2_last.pt (with optimizer state) exist?
import os, sys, time, json, math, argparse
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_cache, minmax_pm1, ctx_of, eps_pred, load_model, main_ckpt, find, OUT

ap = argparse.ArgumentParser()
ap.add_argument("--t", type=int, default=100)
ap.add_argument("--k_dt", type=int, default=1)
ap.add_argument("--p", type=float, default=0.0)        # 0 = take the best setting from A4_fp.json (else 3e-3)
ap.add_argument("--reps", type=int, default=3)
ap.add_argument("--base_steps", type=int, default=32)
a = ap.parse_args()
TAG = "A5"
dev = torch.device("cuda")
DTAU = 1.0 / 1000
rep = {}


# ======================= FP product loss: plan, residuals, analytic coefficients =======================
def plan(x, t, u, v1, v2, p, k):
    hu, hv = 256.0 * p, p
    E = [("tp", x, t + k), ("tm", x, t - k), ("s0", x, t), ("up", x + hu * u, t), ("um", x - hu * u, t)]
    for j, v in ((1, v1), (2, v2)):
        E += [(f"pp{j}", x + hu * u + hv * v, t), (f"pm{j}", x + hu * u - hv * v, t),
              (f"mp{j}", x - hu * u + hv * v, t), (f"mm{j}", x - hu * u - hv * v, t)]
    return E


def residuals(s, x, u, v1, v2, p, k, beta):
    hu, hv = 256.0 * p, p
    A = (u * (s["tp"] - s["tm"])).sum() / (2 * k * DTAU)
    Ju = (s["up"] - s["um"]) / (2 * hu)
    base = (u * s["s0"]).sum() + (Ju * (x + 2 * s["s0"])).sum()
    out = []
    for j, v in ((1, v1), (2, v2)):
        D2 = (v * (s[f"pp{j}"] - s[f"pm{j}"] - s[f"mp{j}"] + s[f"mm{j}"])).sum() / (4 * hu * hv)
        out.append(A - 0.5 * beta * (base + D2))
    return out[0], out[1]


def coeffs(s, x, u, v1, v2, p, k, beta, r1, r2, w=1.0):
    """dL/ds_k for L = w r1 r2 (shared evaluations: w (r1 + r2) dr/ds_k; probe-specific: w r_other dr_j/ds_k)"""
    hu, hv = 256.0 * p, p
    b = 0.5 * beta
    Ju = (s["up"] - s["um"]) / (2 * hu)
    shared = {"tp": u / (2 * k * DTAU), "tm": -u / (2 * k * DTAU), "s0": -b * (u + 2 * Ju),
              "up": -b * (x + 2 * s["s0"]) / (2 * hu), "um": b * (x + 2 * s["s0"]) / (2 * hu)}
    g = {key: w * (r1 + r2) * val for key, val in shared.items()}
    for j, v, other in ((1, v1, r2), (2, v2, r1)):
        c = b * v / (4 * hu * hv)
        g[f"pp{j}"], g[f"pm{j}"], g[f"mp{j}"], g[f"mm{j}"] = -w * other * c, w * other * c, w * other * c, -w * other * c
    return g


def evaluate_nograd(sfn, E):
    """all evaluations without graph; the ones that share a timestep are batched"""
    s = {}
    with torch.no_grad():
        for tt in sorted({e[2] for e in E}):
            keys = [e[0] for e in E if e[2] == tt]
            out = sfn(torch.cat([e[1] for e in E if e[2] == tt]), tt)
            for i, kk in enumerate(keys):
                s[kk] = out[i:i + 1]
    return s


def manual_grad(sfn, x, t, u, v1, v2, p, k, beta, group=1, w=1.0):
    E = plan(x, t, u, v1, v2, p, k)
    s = evaluate_nograd(sfn, E)
    with torch.no_grad():
        r1, r2 = residuals(s, x, u, v1, v2, p, k, beta)
        g = coeffs(s, x, u, v1, v2, p, k, beta, r1, r2, w)
    for tt in sorted({e[2] for e in E}):
        grp = [e for e in E if e[2] == tt]
        for i in range(0, len(grp), group):
            part = grp[i:i + group]
            out = sfn(torch.cat([e[1] for e in part]), tt)
            (torch.cat([g[e[0]] for e in part]) * out).sum().backward()
    return float(r1), float(r2)


def full_graph(sfn, x, t, u, v1, v2, p, k, beta, w=1.0):
    s = {kk: sfn(inp, tt) for kk, inp, tt in plan(x, t, u, v1, v2, p, k)}
    r1, r2 = residuals(s, x, u, v1, v2, p, k, beta)
    (w * r1 * r2).backward()
    return float(r1), float(r2)


# ======================= 1. unit test on a tiny fp64 network =======================
class Tiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1, self.c2, self.e = nn.Conv2d(1, 8, 3, padding=1), nn.Conv2d(8, 1, 3, padding=1), nn.Linear(1, 8)

    def forward(self, x, t):
        tt = torch.full((x.shape[0], 1), t / 1000.0, dtype=x.dtype, device=x.device)
        h = torch.tanh(self.c1(x) + self.e(tt)[:, :, None, None])
        return self.c2(h * h)


torch.manual_seed(0)
tiny = Tiny().double().to(dev)
xs = torch.randn(1, 1, 16, 16, dtype=torch.float64, device=dev)
us = torch.randn_like(xs); us = us / us.norm()
v1s, v2s = (torch.randint(0, 2, xs.shape, device=dev) * 2 - 1).double(), (torch.randint(0, 2, xs.shape, device=dev) * 2 - 1).double()
sf_t = lambda xb, tt: tiny(xb, tt)
grads = []
for fn in (full_graph, manual_grad):
    tiny.zero_grad(set_to_none=True)
    fn(sf_t, xs, 100, us, v1s, v2s, 1e-2, 1, 1.3)
    grads.append(torch.cat([q.grad.flatten() for q in tiny.parameters()]))
ut = float((grads[0] - grads[1]).norm() / grads[0].norm())
rep["unit_test_rel_diff_manual_vs_autograd"] = ut
rep["unit_test_pass"] = bool(ut < 1e-8)
log(TAG, f"unit test: manual-recompute gradient vs autograd rel. diff = {ut:.2e} -> {'PASS' if ut < 1e-8 else 'FAIL'}")
del tiny

# ======================= real network =======================
path = main_ckpt()
unet, ac, meta = load_model(path, dev)
for q in unet.parameters():
    q.requires_grad_(True)
unet.train()
acf = ac.float()
LOGAB = torch.log(ac)
beta_t = float(-(LOGAB[a.t + 1] - LOGAB[a.t - 1]) / (2 * DTAU))
sig_t = lambda tt: math.sqrt(1.0 - float(ac[tt]))
nparam = sum(q.numel() for q in unet.parameters())
rep["unet_parameters_M"] = nparam / 1e6
log(TAG, f"UNet parameters: {nparam / 1e6:.1f} M | {torch.cuda.get_device_name(0)}")

# ---------------- 2. baseline micro-step (original recipe) ----------------
from torchmetrics.functional.image import multiscale_structural_similarity_index_measure as msssim
_sob = torch.tensor([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=torch.float32)
KX, KY = (_sob[None, None] / 8).to(dev), (_sob.t()[None, None] / 8).to(dev)
def hf_loss(x, y):
    gx = F.conv2d(x, KX, padding=1) - F.conv2d(y, KX, padding=1)
    gy = F.conv2d(x, KY, padding=1) - F.conv2d(y, KY, padding=1)
    return (gx.pow(2) + gy.pow(2)).mean()

xtr, ytr = load_cache("train")
Xg, Yg = torch.from_numpy(xtr).to(dev), torch.from_numpy(ytr).long().to(dev)
opt = torch.optim.Adam(unet.parameters(), lr=1e-5)
scaler = torch.amp.GradScaler("cuda")
gg = torch.Generator(device=dev).manual_seed(0)
state0 = {kk: v.detach().clone() for kk, v in unet.state_dict().items()}   # restored after the benchmark


def base_step(i):
    idx = torch.randint(0, Xg.shape[0], (2,), generator=gg, device=dev)
    x0 = minmax_pm1(Xg[idx].float().unsqueeze(1))
    y = Yg[idx]
    t = torch.randint(0, 1000, (2,), generator=gg, device=dev)
    eps = torch.randn(x0.shape, generator=gg, device=dev)
    drop = torch.rand(2, generator=gg, device=dev) < 0.15
    cond = torch.where(drop, torch.full_like(y, -1), y).float().view(2, 1, 1)
    ab = acf[t].view(-1, 1, 1, 1)
    xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps
    with torch.autocast("cuda", dtype=torch.float16):
        eh = unet(xt, timesteps=t, context=cond)
    eh = eh.float()
    mse = F.mse_loss(eh, eps)
    x0h = ((xt - (1 - ab).sqrt() * eh) / ab.sqrt()).clamp(-1, 1)
    ms = 1.0 - msssim((x0h + 1) / 2, (x0 + 1) / 2, data_range=2.0, gaussian_kernel=False, normalize="relu")
    loss = 0.7 * mse + 0.2 * ms + 0.05 * hf_loss(x0h, x0)
    scaler.scale(loss / 4).backward()
    if (i + 1) % 4 == 0:
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)


for i in range(8):
    base_step(i)
torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
t0 = time.time()
for i in range(a.base_steps):
    base_step(i)
torch.cuda.synchronize()
T_base = (time.time() - t0) / a.base_steps
rep["baseline_micro_step_s"] = T_base
rep["baseline_peak_mem_GB"] = torch.cuda.max_memory_allocated() / 1e9
rep["baseline_epoch_min_est_1240_steps"] = 1240 * T_base / 60
log(TAG, f"baseline micro-step (batch 2): {T_base:.3f} s | peak {rep['baseline_peak_mem_GB']:.1f} GB | "
         f"-> {rep['baseline_epoch_min_est_1240_steps']:.1f} min/epoch (Experiment 2 measured 12.4)")
opt.zero_grad(set_to_none=True)
del opt, scaler
unet.load_state_dict(state0)
del state0
torch.cuda.empty_cache()

# ---------------- 3. FP term on one image ----------------
a4 = None
try:
    a4 = json.load(open(os.path.join(OUT, "A4_fp.json")))
except Exception:
    pass
if a.p > 0:
    P_USE, PREC_USE, src = a.p, "fp16", "command line"
elif a4 is not None:
    ch = a4["precision_choice"]
    PREC_USE = "fp16" if ch["fp16"]["usable"] else "fp32"
    P_USE = float(ch[PREC_USE]["best_setting"].split("_p")[1])
    src = f"A4 ({a4['verdict_estimator']})"
else:
    P_USE, PREC_USE, src = 3e-3, "fp32", "default (A4_fp.json not found)"
rep["fp_setting_used"] = {"p": P_USE, "precision": PREC_USE, "source": src}
log(TAG, f"FP setting: p={P_USE:g}, {PREC_USE} ({src})")

xte, yte = load_cache("test")
x0 = minmax_pm1(torch.from_numpy(xte[:1]).float().unsqueeze(1).to(dev))
ab = float(ac[a.t])
gx = torch.Generator(device=dev).manual_seed(1)
x = math.sqrt(ab) * x0 + math.sqrt(1 - ab) * torch.randn(x0.shape, generator=gx, device=dev)
u = torch.randn(x.shape, generator=gx, device=dev); u = u / u.norm()
v1 = (torch.randint(0, 2, x.shape, generator=gx, device=dev) * 2 - 1).float()
v2 = (torch.randint(0, 2, x.shape, generator=gx, device=dev) * 2 - 1).float()
lab = int(yte[0])


def make_sfn(prec):
    def sfn(xb, tt):
        c = ctx_of([lab] * xb.shape[0], dev)
        return -eps_pred(unet, xb, tt, c, amp=(prec == "fp16")) / sig_t(tt)
    return sfn


fp_rows = {}
for prec in ("fp16", "fp32"):
    sfn = make_sfn(prec)
    for group in (1, 2):
        key = f"{prec}_group{group}"
        try:
            unet.zero_grad(set_to_none=True)
            manual_grad(sfn, x, a.t, u, v1, v2, P_USE, a.k_dt, beta_t, group)          # warm-up
            unet.zero_grad(set_to_none=True)
            torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
            t0 = time.time()
            for _ in range(a.reps):
                manual_grad(sfn, x, a.t, u, v1, v2, P_USE, a.k_dt, beta_t, group)
                unet.zero_grad(set_to_none=True)
            torch.cuda.synchronize()
            fp_rows[key] = {"s": (time.time() - t0) / a.reps, "peak_mem_GB": torch.cuda.max_memory_allocated() / 1e9}
        except torch.cuda.OutOfMemoryError:
            fp_rows[key] = {"s": None, "peak_mem_GB": None, "oom": True}
        unet.zero_grad(set_to_none=True); torch.cuda.empty_cache()
        log(TAG, f"FP term, 13 evaluations, manual recompute {key}: {fp_rows[key]}")
rep["fp_manual_recompute"] = fp_rows

# full-graph autograd at batch 1 (expected: out of memory)
try:
    unet.zero_grad(set_to_none=True)
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    full_graph(make_sfn("fp16"), x, a.t, u, v1, v2, P_USE, a.k_dt, beta_t)
    torch.cuda.synchronize()
    rep["fp_full_graph_fp16"] = {"s": time.time() - t0, "peak_mem_GB": torch.cuda.max_memory_allocated() / 1e9}
except torch.cuda.OutOfMemoryError:
    rep["fp_full_graph_fp16"] = {"oom": True}
except Exception as e:
    rep["fp_full_graph_fp16"] = {"error": repr(e)[:200]}
unet.zero_grad(set_to_none=True); torch.cuda.empty_cache()
log(TAG, f"FP term full-graph autograd (fp16, batch 1): {rep['fp_full_graph_fp16']}")

# ---------------- 4. cost ratio + seed rule ----------------
cands = [v["s"] for kk, v in fp_rows.items() if kk.startswith(PREC_USE) and v.get("s")]
if cands:
    T_fp = min(cands)
    rep["T_fp_s_used"] = T_fp
    rep["cost_ratio"] = {str(k): 1 + T_fp / (k * T_base) for k in (1, 2, 4, 8)}
    r4 = rep["cost_ratio"]["4"]
    rep["seed_rule_k4"] = "5 seeds" if r4 <= 2 else "4 seeds" if r4 <= 3 else "reconsider (cost > 3x)"
    rep["epoch_min_est"] = {str(k): rep["baseline_epoch_min_est_1240_steps"] * v for k, v in
                            ((int(k), v) for k, v in rep["cost_ratio"].items())}
    log(TAG, "cost ratio (FP on one image every k micro-steps): " +
        ", ".join(f"k={k}: {v:.2f}x" for k, v in rep["cost_ratio"].items()) + f" | seed rule at k=4: {rep['seed_rule_k4']}")
else:
    rep["seed_rule_k4"] = "not computed (FP benchmark failed)"

# ---------------- 5. Phase B warm-start file ----------------
lp = find("base2_last.pt")
if lp:
    try:
        ck = torch.load(lp[0], map_location="cpu", weights_only=False, mmap=True)
        rep["base2_last"] = {"path": lp[0], "keys": sorted(ck.keys()), "epoch": ck.get("epoch"), "step": ck.get("step"),
                             "optimizer_states": sum(1 for s in ck["opt"]["state"].values() if s) if "opt" in ck else 0,
                             "size_GB": os.path.getsize(lp[0]) / 1e9}
        del ck
    except Exception as e:
        rep["base2_last"] = {"path": lp[0], "error": repr(e)[:200]}
else:
    rep["base2_last"] = "NOT FOUND (Phase B would warm-start from base2_w_ep040.pt weights with a fresh optimizer)"
log(TAG, f"Phase B warm start file: {rep['base2_last']}")
save_json(rep, "A5_bench.json")
''')
Path("report.py").write_text(r'''# report.py - collects the Phase A JSON files into one plain-text report (phaseA_report.txt).
# It only REPORTS the pre-stated rules of each script; the LOCK decisions are written afterwards, by hand, with the RA.
import os, sys, json
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import OUT

L = []
def P(*m):
    L.append(" ".join(str(x) for x in m))
def J(name):
    try:
        return json.load(open(os.path.join(OUT, name)))
    except Exception:
        return None
f = lambda v, d=3: "n/a" if v is None else (f"{v:.{d}f}" if isinstance(v, (int, float)) else str(v))

P("=" * 100); P("PHASE A REPORT (diagnostics only, no training)"); P("=" * 100)

# ---------------- A1 ----------------
r = J("A1_leakage.json")
P("\n[A1] LEAKAGE / DUPLICATES (hard gate)")
if r is None:
    P("  missing (A1 did not finish - see logs/A1.log)")
else:
    P(f"  VERDICT: {r['verdict']}")
    P(f"  patients train/test/both: {r['patients_train']} / {r['patients_test']} / {r['patients_in_both']}"
      f"   images in both splits: {r['images_in_both_splits']}   test-train near-duplicate pairs (r>0.995): "
      f"{r['test_train_pairs_corr_gt_0.995']} (exactly equal: {r['test_train_pairs_exact_equal']})")
    P(f"  cache sizes train/test: {r['cache_train_n']} / {r['cache_test_n']}   test->train max-corr quantiles: "
      f"{ {k: round(v, 4) for k, v in r['test_to_train_max_corr_quantiles'].items()} }")
    P(f"  train rows / unique images / repeats / label conflicts: {r['train_rows']} / {r['train_unique_images']} / "
      f"{r['train_rows_that_repeat_an_image']} / {r['train_images_with_conflicting_labels']}")
    P(f"  test  rows / unique images / repeats / label conflicts: {r['test_rows']} / {r['test_unique_images']} / "
      f"{r['test_rows_that_repeat_an_image']} / {r['test_images_with_conflicting_labels']}")
    P(f"  train-cache duplicate pairs (r>0.995): {r['train_cache_pairs_corr_gt_0.995']}  of which different labels: "
      f"{r['train_cache_dup_pairs_with_different_labels']}")
    for w in r.get("warnings", []):
        P("  WARNING:", w)

# ---------------- A2 ----------------
for which in ("main", "25"):
    r = J(f"A2_jensen_{which}.json")
    P(f"\n[A2-{which}] CONDITIONAL-MEAN (JENSEN) TEST on the real network" + ("" if which == "main" else " (25 % data)"))
    if r is None:
        P("  missing" + (" (no 25 %-data checkpoint attached, or A2-25 not run)" if which == "25" else " (see logs/A2-main.log)"))
        continue
    P(f"  checkpoint {os.path.basename(r['checkpoint'])} epoch {r['epoch']} | {r['n_img']} test images | real L4 band "
      f"[{r['real_band_L4'][0]:.3f}, {r['real_band_L4'][1]:.3f}] | real out-of-band share {r['f_out_real_all']:.2f}")
    P(f"  t* = {r['t_star']}   (rule: x0_hat out-of-band share <= {r['threshold']:.2f} for every grid t up to t*)")
    P("     t | out-of-band share: x0_hat  Bastek  samples  real16 | L4 gap vs real (SD): x0_hat  samples | "
      "Jensen gap S(x0_hat)-mean S(sample) (SD): L1   L2   L3   L4 | K")
    for w in r["rows"]:
        jg = w["jensen_S_of_mean_minus_mean_of_S_sd"]
        P(f"  {w['t']:4d} |   {w['out_band_xhat']:.2f}    {w['out_band_bastek']:.2f}    {w['out_band_samples']:.2f}    "
          f"{w['out_band_real_subset']:.2f} |   {w['gap_sd_xhat'][3]:+.2f}   {w['gap_sd_samples'][3]:+.2f} | "
          f"{jg[0]:+.2f} {jg[1]:+.2f} {jg[2]:+.2f} {jg[3]:+.2f} | {w.get('k_samples', '')}")

# ---------------- A3 ----------------
r = J("A3_texture.json")
P("\n[A3] NOISE vs TEXTURE + LESION SIZE (CPU)")
if r is None:
    P("  missing (see logs/A3.log)")
else:
    P(f"  native JPEG size (median WxH): {r['native_size_median_WxH']}  downsample factor to 256: "
      f"{[round(v, 1) for v in r['downsample_factor_to_256_median']]}")
    P(f"  resolution test ratios (median): { {k: round(v, 3) for k, v in r['resolution_ratios_median'].items()} }")
    P(f"  RESOLUTION VERDICT (flat 256/512; <=0.35 noise-like, >=0.70 texture): {r['resolution_verdict']}")
    P(f"  edge/flat high-pass variance (median over bins): {f(r['edge_over_flat_variance_median'])}")
    P(f"  calibration: high-pass attenuation expected {r['calibration_highpass_attenuation_expected']:.3f} | "
      f"recovered/expected {r['calibration_recovered_over_expected_median']:.3f} | "
      f"texture leak clean/known {r['calibration_texture_leak_clean_over_known_median']:.3f}")
    P(f"  lesion size at 256 px: {r['lesion_size_at_256']}")

# ---------------- A4 ----------------
r = J("A4_fp.json") or J("A4_fp_partial.json")
P("\n[A4] FOKKER-PLANCK RESIDUAL ESTIMATOR on the real network")
if r is None:
    P("  missing (see logs/A4.log)")
else:
    if r.get("partial"):
        P("  !! PARTIAL (A4 stopped early) - rows below are the finished t values only")
    P(f"  reference: {r['reference']}")
    if "verdict_estimator" in r:
        P(f"  ESTIMATOR VERDICT: {r['verdict_estimator']}")
        P(f"  FP residual measurable (z>3) at t = {r['t_with_z_gt_3']} | all t in [10,300]: "
          f"{r['fp_residual_measurable_all_t_10_300']}")
        for prec, c in r["precision_choice"].items():
            P(f"  {prec}: best {c['best_setting']} worst rel.err (t 10-300) {c['worst_rel_err_t10_300']:.3f} usable={c['usable']}"
              f" | all: { {k: round(v, 3) for k, v in c['all_settings_worst_rel_err'].items()} }")
        for c in r.get("fp64_crosscheck", []):
            P(f"  fp64 cross-check: {c}")
    P("     t |     z    rho | term RMS: LHS     u.s      J-term   D2-term | rel.err fp32 p=1e-3 3e-3 1e-2 | "
      "fp16 p=1e-3 3e-3 1e-2 | dtau k=2,5 | guided/unguided")
    for w in r["rows"]:
        m = w["methods"]
        e32 = " ".join(f"{m[k]['rel_err_r']:.3f}" for k in m if k.startswith("fp32"))
        e16 = " ".join(f"{m[k]['rel_err_r']:.3f}" for k in m if k.startswith("fp16"))
        tr = w["term_rms"]
        dk = " ".join(f"{v:.3f}" for v in w["dtau_step_rel_change_vs_k1"].values())
        P(f"  {w['t']:4d} | {w['z']:6.2f} {w['relative_residual_rho']:.3f} | {tr['L']:.2e} {tr['us']:.2e} {tr['Jt']:.2e} "
          f"{tr['D2']:.2e} | {e32} | {e16} | {dk} | {w['guided']['rms_ratio_guided_over_unguided']:.2f}")

# ---------------- A5 ----------------
r = J("A5_bench.json")
P("\n[A5] COMPUTE BENCHMARK")
if r is None:
    P("  missing (see logs/A5.log)")
else:
    P(f"  unit test manual-recompute vs autograd: rel diff {r['unit_test_rel_diff_manual_vs_autograd']:.1e} "
      f"-> {'PASS' if r['unit_test_pass'] else 'FAIL'}")
    P(f"  UNet {r['unet_parameters_M']:.1f} M parameters | baseline micro-step {r['baseline_micro_step_s']:.3f} s, "
      f"peak {r['baseline_peak_mem_GB']:.1f} GB, ~{r['baseline_epoch_min_est_1240_steps']:.1f} min/epoch")
    P(f"  FP setting used: {r['fp_setting_used']}")
    for k, v in r["fp_manual_recompute"].items():
        P(f"  FP term (1 image, 13 evaluations) manual recompute {k}: {v}")
    P(f"  FP term full-graph autograd fp16: {r['fp_full_graph_fp16']}")
    if "cost_ratio" in r:
        P(f"  cost ratio (FP on 1 image every k micro-steps): { {k: round(v, 2) for k, v in r['cost_ratio'].items()} }")
        P(f"  est. min/epoch: { {k: round(v, 1) for k, v in r['epoch_min_est'].items()} }")
    P(f"  SEED RULE (k=4): {r['seed_rule_k4']}")
    P(f"  Phase B warm-start file: {r['base2_last']}")

P("\n" + "=" * 100)
P("Send this whole file (or phaseA_results.zip) back to Claude. Nothing has been decided yet: the LOCK is written after.")
txt = "\n".join(L)
open(os.path.join(OUT, "phaseA_report.txt"), "w").write(txt)
print(txt)
''')
print("wrote phaseA_common.py, pinn_physics.py, a1_leakage.py, a2_jensen.py, a3_texture.py, a4_fp.py, a5_bench.py, report.py")

# ---------- data cache (identical to Experiment 2: 100 % CBIS-DDSM train, 1300 benign + all malignant; official test) ----------
Path("/tmp/prepare.py").write_text(r'''
import os, subprocess
import numpy as np, pandas as pd
from PIL import Image
if os.path.exists("/tmp/train_cache.npz") and os.path.exists("/tmp/test_cache.npz"):
    print("caches already built"); raise SystemExit(0)
hit = subprocess.run("find /kaggle/input -maxdepth 6 -name dicom_info.csv", shell=True,
                     capture_output=True, text=True).stdout.split()
if not hit:
    raise SystemExit("CBIS-DDSM not attached: Add Input -> CBIS-DDSM Breast Cancer Image Dataset")
RAW = os.path.dirname(os.path.dirname(hit[0]))
info = pd.read_csv(os.path.join(RAW, "csv/dicom_info.csv"))
lookup = info[info["SeriesDescription"] == "full mammogram images"].set_index("PatientName")["image_path"].to_dict()
for split, csvs in [("train", ["csv/calc_case_description_train_set.csv", "csv/mass_case_description_train_set.csv"]),
                    ("test",  ["csv/calc_case_description_test_set.csv",  "csv/mass_case_description_test_set.csv"])]:
    xs, skipped = {0: [], 1: []}, 0
    for c in csvs:
        for _, row in pd.read_csv(os.path.join(RAW, c)).iterrows():
            lab = 1 if row["pathology"] == "MALIGNANT" else 0
            rel = lookup.get(row["image file path"].split("/")[0])
            if rel is None:
                skipped += 1; continue
            p = os.path.join(RAW, "jpeg", rel.replace("CBIS-DDSM/jpeg/", ""))
            if not os.path.exists(p):
                skipped += 1; continue
            xs[lab].append(np.array(Image.open(p).convert("L").resize((256, 256)), dtype=np.uint8))
    if split == "train" and len(xs[0]) > 1300:          # same balancing as before: 1300 benign, all malignant
        keep = np.sort(np.random.default_rng(42).choice(len(xs[0]), 1300, replace=False))
        xs[0] = [xs[0][i] for i in keep]
    x = np.stack(xs[0] + xs[1]); y = np.array([0] * len(xs[0]) + [1] * len(xs[1]), dtype=np.int64)
    np.savez(f"/tmp/{split}_cache.npz", x=x, y=y)
    print(f"{split}: {x.shape} benign={int((y == 0).sum())} malignant={int((y == 1).sum())} skipped={skipped}")
''')
subprocess.run([sys.executable, "/tmp/prepare.py"], check=True)

# ---------- inputs ----------
_IDX = {}
for dp, dns, fns in os.walk("/kaggle/input"):                    # one walk; the image folders are skipped
    dns[:] = [d for d in dns if d != "jpeg"]
    for fn in fns:
        _IDX.setdefault(fn, []).append(os.path.join(dp, fn))
def find_in(name):
    return sorted(_IDX.get(name, []))
MAIN = find_in("base2_w_ep040.pt")
LAST = find_in("base2_last.pt")
C25 = [f for s in range(42, 49) for f in find_in(f"base3_s{s}_w_ep045.pt")]
print("main checkpoint  base2_w_ep040.pt :", MAIN[0] if MAIN else
      "!! NOT FOUND -> Add Input -> Your Work -> notebook64a7f79592 (Experiment 2). A2, A4, A5 will FAIL without it.")
print("warm-start file  base2_last.pt    :", LAST[0] if LAST else "not found (only reported by A5)")
print("25 % checkpoint (optional)        :", C25[0] if C25 else "not attached -> A2-25 skipped")
print(subprocess.run("nvidia-smi --query-gpu=index,name,memory.used,memory.total --format=csv",
                     shell=True, capture_output=True, text=True).stdout)
r = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.device_count())"], capture_output=True, text=True)
try:
    NGPU = int(r.stdout.split()[-1])
except Exception:
    NGPU = 0
    print("!! torch check failed:", r.stderr[-300:])
print("GPUs visible:", NGPU, "" if NGPU >= 2 else "!! choose GPU T4 x2 (with 1 GPU the GPU tests run one after the other)")
LOGS = "/kaggle/working/phaseA/logs"
os.makedirs(LOGS, exist_ok=True)
RC = {}

def run_jobs(jobs, poll_min=20):
    """jobs = [(name, gpu_or_None, [script, args...])] -> all run at the same time; output on screen + logs/<name>.log.
    CPU jobs: gpu = None. With fewer than 2 GPUs the GPU jobs run one after the other."""
    if NGPU < 2:
        gpu_jobs = [j for j in jobs if j[1] is not None]
        if len(gpu_jobs) > 1:
            first = [j for j in jobs if j[1] is None] + gpu_jobs[:1]
            run_jobs(first, poll_min)
            for j in gpu_jobs[1:]:
                run_jobs([j], poll_min)
            return
    procs = []
    for name, gpu, argv in jobs:
        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"
        if gpu is None:
            env["CUDA_VISIBLE_DEVICES"] = ""
            env["OMP_NUM_THREADS"] = "2"
        else:
            env["CUDA_VISIBLE_DEVICES"] = str(gpu if NGPU >= 2 else 0)
        fh = open(f"{LOGS}/{name}.log", "w")
        p = subprocess.Popen([sys.executable, "-u"] + argv, cwd=WD, env=env, text=True, bufsize=1,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        procs.append((name, p, fh))
        print(f">>> started {name} on {'CPU' if gpu is None else 'GPU ' + env['CUDA_VISIBLE_DEVICES']} | session {used_h():.2f} h")
    def pump(name, p, fh):
        for line in p.stdout:
            fh.write(line); fh.flush()
            print(line if line.startswith("[") else f"[{name}] {line}", end="", flush=True)
    def poll():
        while any(p.poll() is None for _, p, _ in procs):
            time.sleep(poll_min * 60)
            if any(p.poll() is None for _, p, _ in procs):
                q = subprocess.run("nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader",
                                   shell=True, capture_output=True, text=True).stdout.strip().replace("\n", " | ")
                run = [n for n, p, _ in procs if p.poll() is None]
                print(f"[nvidia-smi] {q} | still running: {run} | session {used_h():.2f} h", flush=True)
    ths = [threading.Thread(target=pump, args=(n, p, fh)) for n, p, fh in procs]
    for th in ths: th.start()
    threading.Thread(target=poll, daemon=True).start()
    for th in ths: th.join()
    for name, p, fh in procs:
        RC[name] = p.wait(); fh.close()
        print(f"<<< {name} finished with return code {RC[name]}" + ("" if RC[name] == 0 else "  !! FAILED -> see " + f"{LOGS}/{name}.log"))
print("Cell 3 done.")

In [ ]:
# ===================== CELL 4 — RUN Phase A (stage 1: A1 + A3 on CPU, A4 on GPU 0, A2 on GPU 1; stage 2: A5 + A2-25) =====================
# A failure of one test does not stop the others. Each test saves its own JSON in /kaggle/working/phaseA.
stage1 = []
if RUN["A1"]: stage1.append(("A1", None, ["a1_leakage.py"]))
if RUN["A3"]: stage1.append(("A3", None, ["a3_texture.py"]))
if RUN["A4"]: stage1.append(("A4", 0, ["a4_fp.py"]))
if RUN["A2"]: stage1.append(("A2-main", 1, ["a2_jensen.py", "--which", "main"]))
t0 = time.time()
run_jobs(stage1)
print(f"===== stage 1 done in {(time.time() - t0) / 3600:.2f} h =====")
stage2 = []
if RUN["A5"]: stage2.append(("A5", 0, ["a5_bench.py"]))          # after A4: uses the precision / step chosen by A4
if RUN["A2"] and RUN_A2_25 and C25: stage2.append(("A2-25", 1, ["a2_jensen.py", "--which", "25"]))
t0 = time.time()
run_jobs(stage2)
print(f"===== stage 2 done in {(time.time() - t0) / 3600:.2f} h | return codes {RC} | session {used_h():.2f} h =====")

In [ ]:
# ===================== CELL 5 — REPORT + zip (send phaseA/phaseA_report.txt or phaseA_results.zip to Claude) =====================
import shutil
r = subprocess.run([sys.executable, "report.py"], cwd=WD, capture_output=True, text=True)
print(r.stdout[-30000:])
if r.returncode != 0:
    print("!! report.py failed:", r.stderr[-1500:])
with open("/kaggle/working/phaseA/return_codes.json", "w") as fh:
    json.dump(RC, fh, indent=1)
z = shutil.make_archive("/kaggle/working/phaseA_results", "zip", "/kaggle/working/phaseA")
print("return codes:", RC)
print("ZIP:", z, f"({os.path.getsize(z) / 1e6:.1f} MB) | report: /kaggle/working/phaseA/phaseA_report.txt | session {used_h():.2f} h")